In [12]:
import os
import tarfile
import random
import re
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torchaudio.utils import _download_asset
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from jiwer import wer
from tqdm.auto import tqdm
import IPython.display as ipd
import whisper

In [13]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

In [14]:
class Config:
    sr = 16000
    batch_size = 8
    epochs = 50
    lr = 1e-4
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    tar_path = "../data/raw/ru_train_0.tar"
    tsv_path = "../data/raw/train(1).tsv"
    extract_dir = "../data/raw/ru_train_data"
    
    n_fft = 512
    hop_length = 256
    win_length = 512

def clean_text(text):
    return re.sub(r'[^\w\s]', '', str(text).lower()).strip()

In [15]:
if not os.path.exists(Config.extract_dir):
    os.makedirs(Config.extract_dir, exist_ok=True)
    with tarfile.open(Config.tar_path, "r") as tar: 
        tar.extractall(path=Config.extract_dir)

df_train = pd.read_csv(Config.tsv_path, sep='\t')
reference_dict = {row['path']: clean_text(row['sentence']) for _, row in df_train.iterrows()}

import soundfile as sf
import torch

babble_path = _download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-babb-mc01-stu-clo-8000hz.wav")
global BABBLE_WAVEFORM

waveform_np, sr_b = sf.read(babble_path, always_2d=True)
BABBLE_WAVEFORM = torch.tensor(waveform_np.T, dtype=torch.float32)
BABBLE_WAVEFORM = T.Resample(sr_b, Config.sr)(BABBLE_WAVEFORM.mean(dim=0, keepdim=True))

rir_path = _download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-impulse-mc01-stu-clo-8000hz.wav")
global RIR_WAVEFORM

waveform_r_np, sr_r = sf.read(rir_path, always_2d=True)
RIR_WAVEFORM = torch.tensor(waveform_r_np.T, dtype=torch.float32)

RIR_WAVEFORM = T.Resample(sr_r, Config.sr)(RIR_WAVEFORM.mean(dim=0, keepdim=True))
RIR_WAVEFORM = RIR_WAVEFORM[:, :int(Config.sr * 0.3)]
RIR_WAVEFORM = RIR_WAVEFORM / torch.norm(RIR_WAVEFORM, p=2)

In [16]:
def get_snr_scale(signal, noise, snr_db):
    sig_power = signal.norm(p=2)**2 / (signal.numel() + 1e-8)
    noise_power = noise.norm(p=2)**2 / (noise.numel() + 1e-8)
    target_noise_power = sig_power / (10 ** (snr_db / 10))
    return torch.sqrt(target_noise_power / (noise_power + 1e-8))

def apply_noise(clean, force_type=None, file_seed=None):
    if file_seed is not None:
        random.seed(file_seed)
        np.random.seed(file_seed)
        torch.manual_seed(file_seed)
        
    snr = random.uniform(-5, 15)
    n_len = clean.shape[-1]

    allowed_noises = ['babble', 'rir', 'white']

    if force_type is not None and force_type in allowed_noises:
        noise_cat = force_type
    else:
        noise_cat = random.choice(allowed_noises)

    if noise_cat == 'babble':
        noise = BABBLE_WAVEFORM
        if noise.shape[-1] < n_len:
            repeats = (n_len // noise.shape[-1]) + 2
            noise = noise.repeat(1, repeats)
        max_start = noise.shape[-1] - n_len
        start = random.randint(0, max_start) if max_start > 0 else 0
        noise_crop = noise[:, start:start+n_len]
        scale = get_snr_scale(clean, noise_crop, snr)
        noisy = clean + noise_crop * scale

    elif noise_cat == 'rir':
        rir = RIR_WAVEFORM
        n_fft_conv = n_len + rir.shape[-1] - 1
        clean_fft = torch.fft.rfft(clean, n=n_fft_conv)
        rir_fft = torch.fft.rfft(rir, n=n_fft_conv)
        augmented = torch.fft.irfft(clean_fft * rir_fft, n=n_fft_conv)
        noisy = augmented[:, :n_len]
        white = torch.randn_like(clean)
        scale = get_snr_scale(noisy, white, snr + 10)
        noisy = noisy + white * scale

    elif noise_cat == 'white':
        noise = torch.randn(1, n_len, device=clean.device)
        noise = noise / (noise.abs().max() + 1e-8)
        scale = get_snr_scale(clean, noise, snr)
        noisy = clean + noise * scale

    max_val = noisy.abs().max()
    if max_val > 1.0:
        noisy = noisy / (max_val + 1e-8)

    return noisy

In [17]:
class ERBFilterBank(nn.Module):
    def __init__(self, n_fft=512, sr=16000, n_erb=32):
        super().__init__()
        self.n_fft = n_fft
        self.n_erb = n_erb
        freqs = np.linspace(0, sr / 2, n_fft // 2 + 1)
        erb_freqs = self._hz_to_erb(freqs)
        
        erb_centers = np.linspace(erb_freqs[0], erb_freqs[-1], n_erb)
        fb = np.zeros((n_erb, n_fft // 2 + 1))
        for i in range(n_erb):
            lower = erb_centers[i-1] if i > 0 else erb_freqs[0]
            upper = erb_centers[i+1] if i < n_erb - 1 else erb_freqs[-1]
            center = erb_centers[i]
            
            fb[i] = np.interp(freqs, 
                              self._erb_to_hz(np.array([lower, center, upper])), 
                              np.array([0, 1, 0]))
            
        self.register_buffer("fb", torch.FloatTensor(fb))

    def _hz_to_erb(self, hz): return 21.4 * np.log10(1 + hz / 229.0)
    def _erb_to_hz(self, erb): return 229.0 * (10**(erb / 21.4) - 1)

    def forward(self, spec_mag):
        return torch.matmul(self.fb, spec_mag)

class DeepFilteringLayer(nn.Module):
    def __init__(self, order=5):
        super().__init__()
        self.order = order

    def forward(self, X, coefs):
        B, _, freqs, T = X.shape
        X_padded = F.pad(X, (self.order - 1, 0))
        X_unfolded = X_padded.unfold(3, self.order, 1) 
        C_re = coefs[..., 0]
        C_im = coefs[..., 1]
        
        X_re = X_unfolded[:, 0]
        X_im = X_unfolded[:, 1]
        
        out_re = torch.sum(X_re * C_re - X_im * C_im, dim=-1)
        out_im = torch.sum(X_re * C_im + X_im * C_re, dim=-1)
        
        return torch.stack([out_re, out_im], dim=1)

In [18]:
class GLinear(nn.Module):
    def __init__(self, in_f, out_f, groups=4):
        super().__init__()
        self.op = nn.Conv1d(in_f, out_f, kernel_size=1, groups=groups)

    def forward(self, x):
        return self.op(x.transpose(1, 2)).transpose(1, 2)

class DF2_Encoder(nn.Module):
    def __init__(self, erb_dim=32, freq_bins=257, hidden_dim=256):
        super().__init__()
        self.in_conv_erb = nn.Sequential(
            nn.ConstantPad2d((1, 1, 2, 0), 0),
            nn.Conv2d(1, 16, kernel_size=(3, 3))
        )
        self.conv1x3_erb = nn.Conv2d(16, 32, kernel_size=(1, 3), padding=(0, 1))

        self.in_conv_cplx = nn.Sequential(
            nn.ConstantPad2d((1, 1, 2, 0), 0),
            nn.Conv2d(2, 16, kernel_size=(3, 3))
        )
        self.conv1x3_cplx = nn.Conv2d(16, 32, kernel_size=(1, 3), padding=(0, 1))

        self.glinear = GLinear(32 * (erb_dim + freq_bins), hidden_dim, groups=4)
        self.gru = nn.GRU(hidden_dim, hidden_dim, num_layers=1, batch_first=True)

    def forward(self, erb, cplx):
        e = F.elu(self.in_conv_erb(erb.transpose(2, 3)))
        e = F.elu(self.conv1x3_erb(e))
        B, C, F_e, T = e.shape
        e = e.permute(0, 3, 1, 2).reshape(B, T, -1)

        c = F.elu(self.in_conv_cplx(cplx.transpose(2, 3)))
        c = F.elu(self.conv1x3_cplx(c))
        B, C, F_c, T = c.shape
        c = c.permute(0, 3, 1, 2).reshape(B, T, -1)

        x = torch.cat([e, c], dim=-1)

        x = F.elu(self.glinear(x))
        out, _ = self.gru(x)
        return out

class DF2_ERB_Decoder(nn.Module):
    def __init__(self, erb_dim=32, hidden_dim=256):
        super().__init__()
        self.gru = nn.GRU(hidden_dim, hidden_dim, num_layers=2, batch_first=True)
        self.pconv = nn.Conv2d(1, 1, kernel_size=(1, 3), padding=(0, 1), groups=1)
        self.glinear = GLinear(hidden_dim, hidden_dim, groups=4)
        self.out = nn.Linear(hidden_dim, erb_dim)

    def forward(self, x):
        B, T, D = x.shape
        g, _ = self.gru(x)
        p = self.pconv(g.unsqueeze(1)).squeeze(1)
        x = F.elu(self.glinear(p))
        gains = torch.sigmoid(self.out(x))
        return gains, p

class DF2_DF_Decoder(nn.Module):
    def __init__(self, freq_bins=257, hidden_dim=256, order=5):
        super().__init__()
        self.order = order
        self.freq_bins = freq_bins
        self.gru = nn.GRU(hidden_dim, hidden_dim, num_layers=2, batch_first=True)
        self.out = GLinear(hidden_dim, freq_bins * order * 2, groups=1)

    def forward(self, enc_out, pconv_out):
        x = enc_out + pconv_out
        x, _ = self.gru(x)
        x = self.out(x)
        
        B, T, _ = x.shape
        x = x.view(B, T, self.freq_bins, self.order, 2).transpose(1, 2)
        return x

In [19]:
class DeepFilterNet2(nn.Module):
    def __init__(self, n_fft=512, sr=16000, n_erb=32):
        super().__init__()
        self.n_fft = n_fft
        self.freq_bins = n_fft // 2 + 1
        self.order = 5
        
        self.erb_bank = ERBFilterBank(n_fft, sr, n_erb)
        self.encoder = DF2_Encoder(n_erb, self.freq_bins)
        self.erb_dec = DF2_ERB_Decoder(n_erb)
        self.df_dec = DF2_DF_Decoder(self.freq_bins, order=self.order)
        self.df_layer = DeepFilteringLayer(order=self.order)

    def forward(self, X_complex):
        mag = torch.abs(X_complex)
        erb_feat = self.erb_bank(mag).log1p()
        
        erb_in = erb_feat.permute(0, 2, 1).unsqueeze(1)
        cplx_in = torch.stack([X_complex.real, X_complex.imag], dim=1).transpose(2, 3)
        
        enc_out = self.encoder(erb_in, cplx_in)
        
        erb_gains, pconv_out = self.erb_dec(enc_out)
        
        lin_gains = torch.matmul(self.erb_bank.fb.T, erb_gains.transpose(1, 2))
        X_stage1 = X_complex * lin_gains
        
        df_coefs = self.df_dec(enc_out, pconv_out)
        
        X_s1_split = torch.stack([X_stage1.real, X_stage1.imag], dim=1)
        
        S_hat_split = self.df_layer(X_s1_split, df_coefs)
        
        return torch.complex(S_hat_split[:, 0], S_hat_split[:, 1])

In [20]:
class SpeechEnhancementDataset(Dataset):
    def __init__(self, data_dir, ref_dict, is_train=True, max_len_sec=3.0):
        self.data_dir = data_dir
        self.ref_dict = ref_dict
        self.is_train = is_train
        self.max_len_sec = max_len_sec
        self.files = []
        for root, _, files in os.walk(data_dir):
            for f in files:
                if f.endswith('.mp3') and f in ref_dict:
                    self.files.append(os.path.join(root, f))

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        filename = os.path.basename(file_path)
        text = self.ref_dict[filename]

        wav_np, sr = librosa.load(file_path, sr=None, mono=False)
        waveform = torch.from_numpy(wav_np)
        if waveform.ndim == 1:
            waveform = waveform.unsqueeze(0)
            
        if sr != Config.sr:
            waveform = T.Resample(sr, Config.sr)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if self.is_train:
            max_samples = int(self.max_len_sec * Config.sr)
            if waveform.shape[-1] > max_samples:
                start = random.randint(0, waveform.shape[-1] - max_samples)
                waveform = waveform[:, start:start + max_samples]
            else:
                pad_len = max_samples - waveform.shape[-1]
                waveform = F.pad(waveform, (0, pad_len))
        
        clean = waveform
        noisy = apply_noise(clean) if self.is_train else clean

        return noisy.squeeze(0), clean.squeeze(0), text

def collate_fn(batch):
    noisy_list, clean_list, texts = zip(*batch)
    return pad_sequence(noisy_list, batch_first=True), \
           pad_sequence(clean_list, batch_first=True), list(texts)

dataset = SpeechEnhancementDataset(Config.extract_dir, reference_dict, is_train=True)
train_size = int(0.9 * len(dataset))
generator = torch.Generator().manual_seed(42)
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, len(dataset)-train_size],generator = generator)
train_loader = DataLoader(train_ds, batch_size=Config.batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=Config.batch_size, shuffle=False, collate_fn=collate_fn)

In [21]:
class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super().__init__()
        self.alpha = alpha

    def forward(self, denoised, clean):
        target = clean - clean.mean(dim=-1, keepdim=True)
        estimate = denoised - denoised.mean(dim=-1, keepdim=True)
        dot = torch.sum(target * estimate, dim=-1, keepdim=True)
        norm = torch.sum(target**2, dim=-1, keepdim=True) + 1e-8
        proj = (dot / norm) * target
        noise = estimate - proj
        si_sdr = -10 * torch.log10((proj**2).sum(-1) / ((noise**2).sum(-1) + 1e-8) + 1e-8)
        
        d_spec = torch.abs(torch.stft(denoised, 512, 256, 512, torch.hann_window(512).to(denoised.device), return_complex=True))
        c_spec = torch.abs(torch.stft(clean, 512, 256, 512, torch.hann_window(512).to(clean.device), return_complex=True))
        spec_loss = F.l1_loss(d_spec, c_spec)
        
        return (1 - self.alpha) * si_sdr.mean() + self.alpha * spec_loss

In [ ]:
model = DeepFilterNet2(n_fft=Config.n_fft, sr=Config.sr).to(Config.device)
criterion = CombinedLoss(alpha=0.2).to(Config.device)
optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr)

window = torch.hann_window(Config.n_fft).to(Config.device)

for epoch in range(1, Config.epochs + 1):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{Config.epochs}")
    
    for noisy, clean, _ in pbar:
        noisy, clean = noisy.to(Config.device), clean.to(Config.device)
        optimizer.zero_grad()
        
        X_complex = torch.stft(noisy, n_fft=Config.n_fft, hop_length=Config.hop_length, 
                               window=window, return_complex=True)
        
        S_hat_complex = model(X_complex)
        
        denoised = torch.istft(S_hat_complex, n_fft=Config.n_fft, hop_length=Config.hop_length, 
                               window=window, length=noisy.shape[-1])
        
        loss = criterion(denoised, clean)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        torch.save(model.state_dict(), f'dfnet2_{epoch}.pth')
        pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

torch.save(model.state_dict(), '../models/dfnet2_weights.pth')

Epoch 1/50:   0%|          | 11/2977 [00:09<43:11,  1.14it/s, Loss=9.6016] 


KeyboardInterrupt: 

In [25]:
model = DeepFilterNet2(n_fft=Config.n_fft, sr=Config.sr).to(Config.device)
criterion = CombinedLoss(alpha=0.2).to(Config.device)
optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr)

window = torch.hann_window(Config.n_fft).to(Config.device)

model.load_state_dict(torch.load('../models/dfnet2_weights.pth', map_location='cpu'))

<All keys matched successfully>

In [ ]:
def evaluate_and_listen(model, device, val_dataset, limit=20):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    
    noise_types = ['babble', 'rir', 'white']
    stats = {n: {"wer_n": [], "wer_d": [], "sisdr_gain": []} for n in noise_types}
    
    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))
        
    window = torch.hann_window(Config.n_fft).to(device)

    with torch.no_grad():
        for idx in tqdm(indices, desc="WER Evaluation"):
            _, clean_wav, ref_text = val_dataset[idx]
            ref_text = clean_text(ref_text)
            
            for n_type in noise_types:
                noisy_wav = apply_noise(clean_wav.unsqueeze(0), force_type=n_type, file_seed=idx).to(device)
                
                X = torch.stft(noisy_wav.squeeze(1), Config.n_fft, Config.hop_length, window=window, return_complex=True)
                S_hat = model(X)
                denoised_wav = torch.istft(S_hat, Config.n_fft, Config.hop_length, window=window, length=noisy_wav.shape[-1])
                
                t_n = asr.transcribe(noisy_wav.squeeze().cpu().numpy(), fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_wav.squeeze().cpu().numpy(), fp16=False, language='ru')['text']
                
                stats[n_type]["wer_n"].append(wer(ref_text, clean_text(t_n)))
                stats[n_type]["wer_d"].append(wer(ref_text, clean_text(t_d)))

    print(f"\n{'Noise Type':<10} | {'WER Noisy':<10} | {'WER Denoised':<10} | {'Gain':<10}")
    for n in noise_types:
        wn, wd = np.mean(stats[n]["wer_n"]), np.mean(stats[n]["wer_d"])
        print(f"{n:<10} | {wn:<10.4f} | {wd:<10.4f} | {wn - wd:<10.4f}")

seed_everything(42)

evaluate_and_listen(model, Config.device, val_ds, limit = 20)

WER Evaluation:   0%|          | 0/20 [00:00<?, ?it/s]


Noise Type | WER Noisy  | WER Denoised | Gain      
babble     | 0.5376     | 0.5208     | 0.0167    
rir        | 1.0000     | 0.9944     | 0.0056    
white      | 0.5670     | 0.6605     | -0.0935   


In [ ]:
from jiwer import process_words
from tqdm.auto import tqdm
import torch
import whisper
import numpy as np

def evaluate_and_listen_components(model, device, val_dataset, limit=None):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    
    noise_types = ['babble', 'rir', 'white']
    
    stats = {n: {
        "wer_n": [], "s_n": [], "d_n": [], "i_n": [],
        "wer_d": [], "s_d": [], "d_d": [], "i_d": []
    } for n in noise_types}
    
    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))
        
    window = torch.hann_window(Config.n_fft).to(device)
    
    with torch.no_grad():
        for idx in tqdm(indices, desc="Evaluation (Macro-average)"):
            _, clean_wav, ref_text = val_dataset[idx]
            clean_tensor_cpu = clean_wav.unsqueeze(0)
    
            for n_type in noise_types:
                noisy_tensor_cpu = apply_noise(clean_tensor_cpu, force_type=n_type, file_seed=idx)
                noisy_tensor = noisy_tensor_cpu.to(device)
                
                X_comp = torch.stft(
                    noisy_tensor.squeeze(1), 
                    n_fft=Config.n_fft, 
                    hop_length=Config.hop_length, 
                    window=window, 
                    return_complex=True
                )
                
                S_hat = model(X_comp)
                
                denoised_tensor = torch.istft(
                    S_hat, 
                    n_fft=Config.n_fft, 
                    hop_length=Config.hop_length, 
                    window=window, 
                    length=noisy_tensor.shape[-1]
                )
    
                noisy_np = noisy_tensor.squeeze().cpu().numpy()
                denoised_np = denoised_tensor.squeeze().cpu().numpy()
    
                t_n = asr.transcribe(noisy_np, fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_np, fp16=False, language='ru')['text']
    
                clean_ref = clean_text(ref_text)
                clean_hyp_n = clean_text(t_n)
                clean_hyp_d = clean_text(t_d)
    
                out_n = process_words(clean_ref, clean_hyp_n)
                n_words_n = out_n.substitutions + out_n.deletions + out_n.hits
                
                if n_words_n > 0:
                    stats[n_type]["wer_n"].append((out_n.substitutions + out_n.deletions + out_n.insertions) / n_words_n)
                    stats[n_type]["s_n"].append(out_n.substitutions / n_words_n)
                    stats[n_type]["d_n"].append(out_n.deletions / n_words_n)
                    stats[n_type]["i_n"].append(out_n.insertions / n_words_n)
                else:
                    stats[n_type]["wer_n"].append(0.0)
                    stats[n_type]["s_n"].append(0.0)
                    stats[n_type]["d_n"].append(0.0)
                    stats[n_type]["i_n"].append(0.0)
    
                out_d = process_words(clean_ref, clean_hyp_d)
                n_words_d = out_d.substitutions + out_d.deletions + out_d.hits
                
                if n_words_d > 0:
                    stats[n_type]["wer_d"].append((out_d.substitutions + out_d.deletions + out_d.insertions) / n_words_d)
                    stats[n_type]["s_d"].append(out_d.substitutions / n_words_d)
                    stats[n_type]["d_d"].append(out_d.deletions / n_words_d)
                    stats[n_type]["i_d"].append(out_d.insertions / n_words_d)
                else:
                    stats[n_type]["wer_d"].append(0.0)
                    stats[n_type]["s_d"].append(0.0)
                    stats[n_type]["d_d"].append(0.0)
                    stats[n_type]["i_d"].append(0.0)
    
    header = f"{'Noise':<8} | {'WER_N':<7} (S/D/I) | {'WER_D':<7} (S/D/I) | {'Gain WER':<8}"
    print(header)
    print("-" * 75)
    
    for n_type in noise_types:
        wer_n = np.mean(stats[n_type]["wer_n"])
        s_n_pct = np.mean(stats[n_type]["s_n"])
        d_n_pct = np.mean(stats[n_type]["d_n"])
        i_n_pct = np.mean(stats[n_type]["i_n"])
    
        wer_d = np.mean(stats[n_type]["wer_d"])
        s_d_pct = np.mean(stats[n_type]["s_d"])
        d_d_pct = np.mean(stats[n_type]["d_d"])
        i_d_pct = np.mean(stats[n_type]["i_d"])
    
        str_noisy = f"{wer_n:.4f} ({s_n_pct:.4f}/{d_n_pct:.4f}/{i_n_pct:.4f})"
        str_denois = f"{wer_d:.4f} ({s_d_pct:.4f}/{d_d_pct:.4f}/{i_d_pct:.4f})"
    
        print(f"{n_type:<8} | {str_noisy:<28} | {str_denois:<28} | {wer_n - wer_d:<8.4f}")

seed_everything(42)
evaluate_and_listen_components(model, Config.device, val_ds, limit=20)

Evaluation (Macro-average):   0%|          | 0/20 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

Noise    | WER_N   (S/D/I) | WER_D   (S/D/I) | Gain WER
---------------------------------------------------------------------------
babble   | 0.5376 (0.1392/0.3875/0.0108) | 0.5208 (0.1440/0.3580/0.0188) | 0.0167  
rir      | 1.0000 (0.3116/0.6884/0.0000) | 0.9944 (0.2121/0.7823/0.0000) | 0.0056  
white    | 0.5670 (0.2060/0.3548/0.0063) | 0.6605 (0.2595/0.3806/0.0204) | -0.0935 
